In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter
# import icd10
from ast import literal_eval

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [2]:
# count number of rows
from itertools import (takewhile,repeat)
def rawincount(filename):
    f = open(filename, 'rb')
    bufgen = takewhile(lambda x: x, (f.raw.read(1024*1024) for _ in repeat(None)))
    return sum( buf.count(b'\n') for buf in bufgen )

### Remove carriage returns for the files

In [ ]:
# DO NOT EXECUTE ANYMORE - ROWS CLEANED OF CARRIAGE RETURN ALREADY
# path, dirs, files = next(os.walk("chunks/"))
# file_count = len(files)
# year_start = 2013
# year_end = 2020

# for i in range(year_start, year_end+1):
#     directory = 'chunks/'
#     print(i)
#     print('total count:', rawincount(os.path.join(directory, str(i) + '_new.csv')))
#     df1 = pd.read_csv(os.path.join(directory, str(i) + '_new.csv'), low_memory=False)
#     df1['ILLNESS1_DESC'] = df1['ILLNESS1_DESC'].str.replace(r'\n','')
#     df1.to_csv(os.path.join(directory, str(i) + '_new_1.csv'), index=False)
#     del[df1]
#     gc.collect()    

### Parm LIb declarations

In [13]:
path, dirs, files = next(os.walk("chunks_3/"))
directory = 'chunks_3/'
file_count = len(files)
year_start = 2014
year_end = 2020

In [4]:
# codes in ILLNESS*_CODE that are not ICD-10 codes
not_icd = ['MCP01', 'NSD01', 'P0000', 'P0001', 'ANC01', 'ANC02', 'FP001',
          'C19CI', 'C19CIS', 'C19FRP', 'C19IP1', 'C19IP2', 'C19IP3', 'C19IP4', 'C19T1', 'C19T2', 'C19T3', 'C19X3',
          'Z0011', 'Z0012', 'Z0013', 'Z0021', 'Z0022', 'Z003', 'Z0041', 'Z0042', 'Z0050', 'Z0051', 'Z0052', 'Z0061',
          'Z0062', 'Z0071', 'Z0072', 'Z0081', 'Z0082', 'Z0091', 'Z0092', 'Z010A', 'Z010B', 'Z010C', 'Z011A', 'Z011B',
          'Z011C', 'Z011D', 'Z011E', 'Z011F', 'Z011G', 'Z011G1', 'Z011G2', 'Z011H', 'Z011H1', 'Z011H2', 'Z011I',
          'Z01201', 'Z01202', 'Z01203', 'Z01204', 'Z01205', 'Z01206', 'Z01207', 'Z01208', 'Z01209', 'Z01210', 'Z01211',
          'Z01212', 'Z01213', 'Z01214', 'Z01215', 'Z01216', 'Z01217', 'Z01218', 'Z01219', 'Z01220', 'Z01221', 'Z01222',
          'Z01223', 'Z01224', 'Z01225', 'Z01226', 'Z0131A', 'Z0131B', 'Z0132B', 'Z0141A', 'Z0141B', 'Z0141CC', 'Z0141CL',
          'Z0142BC', 'Z0142BL', 'Z0142C', 'Z0143B', 'Z0143C', 'Z015101', 'Z0151A1', 'Z0151A2', 'Z0151B1', 'Z0151B2',
          'Z0151C1', 'Z0152A1', 'Z0152A2', 'Z0153A1', 'Z0153C1', 'Z0154A1', 'Z0154B1', 'Z0156A1', 'Z0156A2', 'Z0156B1',
          'Z0156B2', 'Z0156C1', 'Z0156C2', 'Z0157A1', 'Z0157B1', 'Z0157B2', 'Z0157C1', 'Z01591', 'Z0165', 'Z0166', 
          'Z0167', 'Z0168', 'Z0169']

#icd-10 philhealth list - 2017
df_icd10 = pd.read_excel(os.path.join(directory, 'ICD10 philhealth.xlsx'))
df_icd10 = df_icd10.set_axis(['ILLNESS1_CODE','DESCRIPTION','GROUP','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

#procedures philhealth list - 2015
df_procs = pd.read_excel(os.path.join(directory, 'Procedures philhealth.xlsx'))
df_procs = df_procs.set_axis(['ILLNESS1_CODE','DESCRIPTION','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

In [5]:
# 25th Percentile
def q25(x):
    return x.quantile(0.25)

# 75th Percentile
def q75(x):
    return x.quantile(0.75)

# counts per single categorical field
def group_categs(dframe, col_name_grp, ren_cols, tab_name, writer1):
    df2 = dframe.groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
                .set_axis([ren_cols, 'Freq.'], axis=1, inplace=False)
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name, index=False)
    del[df2]

# counts per combination of categorical fields
def sort_val_categs(dframe, col_name_grps, tab_name, writer1):
    de_bracket_cols = '[' + (', '.join('\'' + item + '\'' for item in col_name_grps)) + ', \'Freq.\']'
    df2 = df1.sort_values(col_name_grps,ascending=False).groupby(col_name_grps).size().to_frame().reset_index() \
            .set_axis(literal_eval(de_bracket_cols), axis=1, inplace=False)
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name, index=False)
    del[df2]

# computes for ILLNESS*_CODE total counts
def illness_counts(dframe, \
                   not_in_icd, df_icd, df_proc, \
                   col_name_grp, \
                   tab_name_med, tab_name_proc, \
                   writer1):   
    df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=col_name_grp)[[col_name_grp,'Freq.','DESCRIPTION']]
    df2 = df2[~df2['Freq.'].isnull()]    
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2 = df2.loc[~df2['Freq.'].isnull()] 
    df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)        
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
    df2 = dframe.loc[~(dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd))] \
            .groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Freq.'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=col_name_grp)[[col_name_grp,'Freq.','DESCRIPTION']]
    df2 = df2[~df2['Freq.'].isnull()]      
    df2['Percent'] = ((df2['Freq.'].values/total_year)*100)
    df2['Freq.'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE AMOUNT totals 
def illness_amts(dframe, \
                 not_in_icd, df_icd, df_proc, \
                 col_name_grp, amt_col, 
                 tab_name_med, tab_name_proc, \
                 tot_pd_amt, writer1): 
    df2 = dframe.loc[dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd)] \
            .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=col_name_grp)[[col_name_grp,'Total Amount of Claims','DESCRIPTION']]  
    df2 = df2[~df2['Total Amount of Claims'].isnull()]      
    df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
    df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
                                        .format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]
        
#     df2 = dframe.loc[dframe[col_name_grp].str.isnumeric() | dframe[col_name_grp].isin(not_in_icd)] \
    df2 = dframe.loc[~(dframe[col_name_grp].astype(str).str[0].str.isalpha() & ~dframe[col_name_grp].isin(not_in_icd))] \
            .groupby([col_name_grp])[amt_col].sum().sort_values(ascending=False).to_frame().reset_index() \
            .set_axis([col_name_grp, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=col_name_grp)[[col_name_grp,'Total Amount of Claims','DESCRIPTION']]    
    df2 = df2[~df2['Total Amount of Claims'].isnull()]       
    df2['Percent'] = ((df2['Total Amount of Claims'].values/tot_pd_amt)*100)
    df2['Total Amount of Claims'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}' \
                                        .format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.swifter.progress_bar(False).apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE totals, +1 col grouping besides illness code field
def illness_counts_oth_1(dframe, 
                not_in_icd, df_icd, df_proc, \
                illness_cd_col, col_name_grp, \
                tab_name_med, tab_name_proc, \
                tot_year, writer1):
    df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
            .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Freq.'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Freq.','DESCRIPTION']]  \
            .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])
    df2 = df2[~df2['Freq.'].isnull()]       
    df2['Percent'] = ((df2['Freq.']/tot_year)*100)
    df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)        
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
    df2 = df1.loc[~(df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd))] \
            .groupby([col_name_grp, illness_cd_col]).size().to_frame().reset_index() \
            .set_axis([col_name_grp,illness_cd_col, 'Freq.'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Freq.','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Freq.'], ascending=[True, False])
    df2 = df2[~df2['Freq.'].isnull()]      
    df2['Percent'] = ((df2['Freq.']/tot_year)*100)
    df2['Freq.'] = df2.apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

# computes for ILLNESS*_CODE amounts, +1 col grouping besides illness code field    
def illness_amts_oth_1(dframe, 
                not_in_icd, df_icd, df_proc, \
                illness_cd_col, col_name_grp, amt_col, \
                tab_name_med, tab_name_proc, \
                tot_amt, writer1):    
    df2 = df1.loc[df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd)] \
            .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_icd, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Total Amount of Claims','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])
    df2 = df2[~df2['Total Amount of Claims'].isnull()]        
    df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
    df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_med, index=False)
    del[df2]

#     df2 = df1.loc[df1[illness_cd_col].str.isnumeric() | df1[illness_cd_col].isin(not_icd)] \
    df2 = df1.loc[~(df1[illness_cd_col].astype(str).str[0].str.isalpha() & ~df1[illness_cd_col].isin(not_icd))] \
            .groupby([col_name_grp, illness_cd_col])[amt_col].sum().to_frame().reset_index() \
            .set_axis([col_name_grp, illness_cd_col, 'Total Amount of Claims'], axis=1, inplace=False) \
            .merge(df_proc, how='outer', on=illness_cd_col)[[col_name_grp, illness_cd_col, \
                                                                'Total Amount of Claims','DESCRIPTION']] \
            .sort_values(by=[col_name_grp, 'Total Amount of Claims'], ascending=[True, False])
    df2 = df2[~df2['Total Amount of Claims'].isnull()]      
    df2['Percent'] = ((df2['Total Amount of Claims']/tot_amt)*100)
    df2['Total Amount of Claims'] = df2.apply(lambda x: '{:,.2f}'.format(x['Total Amount of Claims']),axis=1)
    df2['Percent'] = df2.apply(lambda x: '{:,.2f}'.format(x['Percent']),axis=1)
    df2.to_excel(writer1, sheet_name=tab_name_proc, index=False)
    del[df2]

def turnaround_time_1(dframe, col_name_grp, tab_name, tab_name_365):   
    df2 = df1.loc[df1['TAT'] >= 0].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name, index=False)
    del[df2]
    del[df3]

    df2 = df1.loc[(df1['TAT'] >= 0) & (df1['TAT'] <= 365)].groupby([col_name_grp])['TAT'] \
            .agg(['mean', 'median', 'std', 'min', 'max', q25, q75, 'count']) 
    df3 = df1.loc[(df1['TAT'] > 60) & (df1['TAT'] <= 365)].groupby([col_name_grp]).size().to_frame().reset_index()
    df3 = df3.set_axis([col_name_grp, 'numgreater60'], axis=1, inplace=False)
    df2 = df2.merge(df3, how='inner', on=col_name_grp)
    df2['%>60days'] = 100*(df2['numgreater60']/df2['count'])
    df2.to_excel(writer, sheet_name=tab_name_365, index=False)
    del[df2]
    del[df3]
    
def unpaid_process(dframe, \
                check_dt_col, \
                rr_col, amt_col, acr_amt_col, \
                agg_col, tab_name, writer1):
#     df2 = dframe[dframe[check_dt_col].isna() & ~dframe[rr_col].isna()]
    df2 = dframe[dframe['CLAIM_STATUS'] == check_dt_col]
    df2[amt_col].fillna(df2[acr_amt_col], inplace=True)    
    df2.groupby([agg_col])[amt_col].agg(['mean', 'median', 'count', 'min', 'max']) \
        .to_excel(writer1, sheet_name=tab_name, index=True)
    del[df2]

In [14]:
for i in range(year_start, year_end+1):
    df_all = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)
    df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
            .drop(columns=['MEMBER_NAME','PATIENT_NAME'])

    # for lighter pandas df and faster execution
    df1['PRO_NAME'] = df1['PRO_NAME'].astype('category')
    df1['INSTITUTION_NAME'] = df1['INSTITUTION_NAME'].astype('category')
    df1['OWNERSHIP'] = df1['OWNERSHIP'].astype('category')
    df1['INSTITUTION_CLASS'] = df1['INSTITUTION_CLASS'].astype('category')
    df1['INSTITUTION_PROVINCE'] = df1['INSTITUTION_PROVINCE'].astype('category')
    df1['INSTITUTION_MUNICIPALITY'] = df1['INSTITUTION_MUNICIPALITY'].astype('category')
    df1['MEM_CATEGORY'] = df1['MEM_CATEGORY'].astype('category')

    print(i)
    print(len(df_all.index))
    total_year = len(df1.index)
    total_paid_amt = df1['PAID AMOUNT'].sum()
#     print('with illness code NaN:', len(df1[df1['ILLNESS1_CODE']].isnull()))
    
    writer = pd.ExcelWriter(os.path.join(directory, str(i) + '_totals.xlsx'), engine='xlsxwriter')

    # Medical and Procedural claims: member cat, illness1_code, illness1 frequency
    illness_counts_oth_1(df1, 
                    not_icd, df_icd10, df_procs, \
                    'ILLNESS1_CODE', 'MEM_CATEGORY', \
                    'Med Claim - Counts-Mem Categ', 'Proc Claim - Counts-Mem Categ', \
                    total_year, writer)    

    # member category and paid amount 
    illness_amts_oth_1(df1, 
                    not_icd, df_icd10, df_procs, \
                    'ILLNESS1_CODE', 'MEM_CATEGORY', 'PAID AMOUNT', \
                    'Med Claim - Amt-Mem Categ', 'Proc Claim - Amt-Mem Categ', \
                    total_paid_amt, writer)
    
    writer.save()  
   
    del[df1]
    del[df_all]
    gc.collect()
    gc.collect()

2014
6829161
2015
9026455
2016
10671227
2017
10904857
2018
11562595
2019
11744401
2020
11226777
